# 规则组 write

保留原规则组；只读取 B站、小红书 `new/` 中能够从 `query` 或 `topic_title` 明确识别的规则组，写出 `rules-groups.csv`。

In [2]:
from pathlib import Path
import re
import pandas as pd

DATA = Path("../数据")
RAW = DATA / "01_原始数据"
X_DATA = DATA / "02_X匹配数据"
PLATFORMS = ["小红书", "B站"]
INPUT_FOLDERS = ["new"]

In [3]:
environment = pd.read_csv(X_DATA / "environment_metadata.csv")
base_groups = environment[["ENV", "trading_platform", "event", "event_year"]].copy()
base_groups

,ENV,trading_platform,event,event_year
0,京东|双11|2022,京东,双11,2022
1,京东|双11|2023,京东,双11,2023
2,淘宝|618|2021,淘宝,618,2021
3,淘宝|双11|2021,淘宝,双11,2021
4,京东|618|2023,京东,618,2023
5,淘宝|618|2023,淘宝,618,2023
6,淘宝|618|2025,淘宝,618,2025
7,淘宝|双11|2025,淘宝,双11,2025
8,抖音|618|2026,抖音,618,2026
9,京东|618|2026,京东,618,2026


In [4]:
def get_group(text):
    text = str(text)

    platform_match = re.search(r"京东|淘宝|天猫|拼多多|抖音", text)
    event_match = re.search(r"618|双\s*(?:11|十一)|11[.．]11", text, re.I)
    year_match = re.search(r"(?<!\d)(20\d{2})(?!\d)", text)
    if year_match is None:
        year_match = re.search(r"(?<!\d)(2[0-9])年", text)

    if not (platform_match and event_match and year_match):
        return None

    trading_platform = platform_match.group()
    if trading_platform == "天猫":
        trading_platform = "淘宝"

    event = "618" if "618" in event_match.group() else "双11"
    year = int(year_match.group(1))
    if year < 100:
        year += 2000

    return trading_platform, event, year


rows = []
input_files = []

for social_platform in PLATFORMS:
    for folder in INPUT_FOLDERS:
        for path in sorted((RAW / social_platform / folder).glob("*.csv")):
            input_files.append(path)
            source = pd.read_csv(path)
            text_column = "query" if "query" in source.columns else "topic_title"
            if text_column not in source.columns:
                continue

            for text in source[text_column].dropna().drop_duplicates():
                group = get_group(text)
                if group:
                    trading_platform, event, event_year = group
                    rows.append({
                        "ENV": f"{trading_platform}|{event}|{event_year}",
                        "trading_platform": trading_platform,
                        "event": event,
                        "event_year": event_year,
                    })

new_groups = pd.DataFrame(rows, columns=base_groups.columns).drop_duplicates()
print("读取的新增 CSV：")
for path in input_files:
    print(path)
new_groups

读取的新增 CSV：
../数据/01_原始数据/小红书/new/抖音2021年双十一_ff558daace95.csv
../数据/01_原始数据/小红书/new/抖音2022年双十一_b75abe85f128.csv
../数据/01_原始数据/小红书/new/抖音_2021_双十一_89850c954651.csv
../数据/01_原始数据/小红书/new/抖音_2022_双十一_99dcc507f8aa.csv
../数据/01_原始数据/小红书/new/抖音_2023_双十一_06e7e51d6e56.csv
../数据/01_原始数据/小红书/new/抖音双十一_58eb34e17700.csv
../数据/01_原始数据/小红书/new/拼多多_2021_双11_961fa093c7b5.csv
../数据/01_原始数据/小红书/new/拼多多_2022_双11_a04942d8188b.csv
../数据/01_原始数据/小红书/new/拼多多_2023_双11_4f565757e98a.csv
../数据/01_原始数据/B站/new/抖音双11_09684811bb11.csv
../数据/01_原始数据/B站/new/抖音双11_246298dce79f.csv
../数据/01_原始数据/B站/new/抖音双11_b293ce625a5d.csv
../数据/01_原始数据/B站/new/抖音双十一_4798de6da88c.csv
../数据/01_原始数据/B站/new/抖音双十一_8613d590c703.csv
../数据/01_原始数据/B站/new/抖音双十一_f5f38d20864a.csv
../数据/01_原始数据/B站/new/拼多多双11_b07f7d9c3f09.csv
../数据/01_原始数据/B站/new/拼多多双11_e8c035270f36.csv
../数据/01_原始数据/B站/new/拼多多双十一_0bb345cc9015.csv
../数据/01_原始数据/B站/new/拼多多双十一_b883df091cd3.csv


,ENV,trading_platform,event,event_year
0,抖音|双11|2022,抖音,双11,2022
1,抖音|双11|2021,抖音,双11,2021
3,抖音|双11|2023,抖音,双11,2023
4,拼多多|双11|2021,拼多多,双11,2021
5,拼多多|双11|2022,拼多多,双11,2022
6,拼多多|双11|2023,拼多多,双11,2023


In [5]:
rules_groups = (
    pd.concat([base_groups, new_groups], ignore_index=True)
    .drop_duplicates(subset="ENV")
    .sort_values(["event_year", "trading_platform", "event"])
    .reset_index(drop=True)
)

output_path = X_DATA / "rules-groups.csv"
rules_groups.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"已写出：{output_path}")
print(f"规则组数量：{len(rules_groups)}")
rules_groups

已写出：../数据/02_X匹配数据/rules-groups.csv
规则组数量：33


,ENV,trading_platform,event,event_year
0,京东|双11|2021,京东,双11,2021
1,抖音|双11|2021,抖音,双11,2021
2,拼多多|双11|2021,拼多多,双11,2021
3,淘宝|618|2021,淘宝,618,2021
4,淘宝|双11|2021,淘宝,双11,2021
5,京东|618|2022,京东,618,2022
6,京东|双11|2022,京东,双11,2022
7,抖音|双11|2022,抖音,双11,2022
8,拼多多|双11|2022,拼多多,双11,2022
9,淘宝|双11|2022,淘宝,双11,2022
